# EXP015 · Frozen EXP013 command-reference residual + Hundman

Select **A100 40 GB** (or A100 80 GB, subject to identical numerical gates) and choose **Runtime → Run all**.

This separately labelled supplement performs **zero training**. It evaluates the selected EXP013 graph, separately fitted removal, and direct-fusion models at seeds 17, 29 and 43 with the exact frozen EXP012 detector. EXP012–EXP014 scientific results and the Overleaf report remain unchanged.

The original EXP013 score exports do not retain the normal-reference predictions. Frozen inference is required. The signed discriminator hybrid is excluded for the prespecified scale/guard compatibility reasons in `docs/SCIENTIFIC_DECISION.md`.

**Required immutable input:** `MyDrive/EXP013_COMMAND_REFERENCE_V1_E03_CONV/main/FINAL_RESULTS.zip`. The notebook reads 18 checkpoints, their metadata/threshold bindings and 729 native score references from this existing archive. The original input bundle or reports ZIP is insufficient. You need upload only this notebook's paired input bundle once, if it is not already cached.

Development verification is CPU/synthetic and includes established detector regression streams. **No real EXP013 checkpoint replay or A100 NASA execution is claimed before this notebook's mandatory gates pass.**


In [ ]:
# 1. Mount Drive, locate originals and verify the exact paired release.
from google.colab import drive, files
from pathlib import Path
import sys,os,json,hashlib,zipfile,shutil,subprocess,uuid,time

RUN_SLOT='main'
SOURCE_PATH=Path('/content/drive/MyDrive/EXP013_COMMAND_REFERENCE_V1_E03_CONV/main/FINAL_RESULTS.zip')
VERSION='EXP015_EXP013_GCN_HUNDMAN_SUPPLEMENT_V1'
BUNDLE_NAME=VERSION+'_input_bundle.zip'
BUNDLE_SHA256='93fd86d725e9549b3910d0aa5959a73e1516bc8292b44af01597d1cc034e0df8'
MANIFEST_SHA256='8d97e47787d94ad6ba670cf9024272737230099d6fd68e1af210ba23ab0d0c9f'
SETUP_STARTED=time.monotonic()
os.environ['MPLBACKEND']='Agg'
BASE_ENV=dict(os.environ,MPLBACKEND='Agg',CUBLAS_WORKSPACE_CONFIG=':4096:8',PYTHONDONTWRITEBYTECODE='1',PYTHONUNBUFFERED='1')
if not RUN_SLOT or any(c not in 'abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789_-' for c in RUN_SLOT):
    raise ValueError('Use letters, numbers, underscore or hyphen for RUN_SLOT.')
if sys.version_info[:2] not in [(3,12),(3,13)]:raise RuntimeError('Python 3.12 or 3.13 required.')
gpu=subprocess.check_output(['nvidia-smi','--query-gpu=name,memory.total,driver_version','--format=csv,noheader,nounits'],text=True,env=BASE_ENV).strip()
print('Notebook Python:',sys.version,'\nGPU (name, MiB, driver):',gpu)
if len(gpu.splitlines())!=1:raise RuntimeError('This notebook expects one A100 GPU.')
fields=gpu.split(',');memory_mib=int(fields[1].strip())
if 'A100' not in fields[0] or not (35000<=memory_mib<=45000 or 75000<=memory_mib<=90000):
    raise RuntimeError('Select an A100 40 GB or 80 GB. No smaller model/batch substitution is allowed.')
drive.mount('/content/drive')
if not SOURCE_PATH.exists():raise FileNotFoundError('Exact immutable file needed: '+str(SOURCE_PATH)+'. Keep the completed EXP013 archive here, or edit SOURCE_PATH to its actual location. No retraining or substituted weights.')
DEST=Path('/content/drive/MyDrive')/VERSION/RUN_SLOT
PACKAGE=Path('/content')/(VERSION+'_package')
WORK=Path('/content')/(VERSION+'_work_'+RUN_SLOT)
CACHE=Path('/content')/(VERSION+'_cache_'+RUN_SLOT)
VENV=Path('/content')/(VERSION+'_venv')
DEST.mkdir(parents=True,exist_ok=True);CACHE.mkdir(parents=True,exist_ok=True)

def sha_file(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for b in iter(lambda:f.read(8*2**20),b''):h.update(b)
    return h.hexdigest()

def persist_verified(src,dst):
    src,dst=map(Path,(src,dst));expected=sha_file(src)
    if dst.exists():
        if sha_file(dst)!=expected:raise RuntimeError('Existing persistent immutable file differs: '+str(dst))
        return
    dst.parent.mkdir(parents=True,exist_ok=True);tmp=dst.with_name(dst.name+'.partial-'+uuid.uuid4().hex)
    try:
        with src.open('rb') as f,tmp.open('wb') as g:shutil.copyfileobj(f,g,8*2**20);g.flush();os.fsync(g.fileno())
        if sha_file(tmp)!=expected:raise RuntimeError('Drive copy readback differs')
        os.replace(tmp,dst)
        if sha_file(dst)!=expected:raise RuntimeError('Drive publication readback differs')
    finally:tmp.unlink(missing_ok=True)

local_bundle=CACHE/BUNDLE_NAME;saved_bundle=DEST/BUNDLE_NAME
if saved_bundle.exists():
    print('Recovering the paired input bundle from Drive.');shutil.copyfile(saved_bundle,local_bundle)
else:
    print('Upload exactly the paired EXP015 input bundle ZIP.');uploaded=files.upload()
    if len(uploaded)!=1:raise RuntimeError('Upload one matching ZIP file.')
    content=next(iter(uploaded.values()))
    if hashlib.sha256(content).hexdigest()!=BUNDLE_SHA256:raise RuntimeError('Wrong input bundle for this notebook.')
    local_bundle.write_bytes(content);del content,uploaded
if sha_file(local_bundle)!=BUNDLE_SHA256:raise RuntimeError('Paired input-bundle hash differs.')
persist_verified(local_bundle,saved_bundle)
if PACKAGE.exists():shutil.rmtree(PACKAGE)
PACKAGE.mkdir(parents=True)
with zipfile.ZipFile(local_bundle) as z:
    if len(z.namelist())!=len(set(z.namelist())):raise RuntimeError('Duplicate bundle paths')
    for item in z.infolist():
        rel=Path(item.filename)
        if rel.is_absolute() or '..' in rel.parts or chr(92) in item.filename or (item.external_attr>>16)&0o170000==0o120000:raise RuntimeError('Unsafe bundle path')
    z.extractall(PACKAGE)
if sha_file(PACKAGE/'MANIFEST.json')!=MANIFEST_SHA256:raise RuntimeError('Pinned manifest differs')
manifest=json.loads((PACKAGE/'MANIFEST.json').read_text())['files']
actual={p.relative_to(PACKAGE).as_posix() for p in PACKAGE.rglob('*') if p.is_file() and p.name!='MANIFEST.json'}
if actual!=set(manifest):raise RuntimeError('Bundle file roster differs')
for n,h in manifest.items():
    if sha_file(PACKAGE/n)!=h:raise RuntimeError('Bundle file hash differs: '+n)
contract=json.loads((PACKAGE/'INPUT_CONTRACT.json').read_text());needed=set(contract['known_required_files'])
if SOURCE_PATH.is_file():
    with zipfile.ZipFile(SOURCE_PATH) as z:missing=needed-set(z.namelist())
else:missing={n for n in needed if not (SOURCE_PATH/n).is_file()}
if missing:raise FileNotFoundError('Original EXP013 source is missing these exact required constituents: '+', '.join(sorted(missing)))
ENV=dict(BASE_ENV,PYTHONPATH=str(PACKAGE/'src')+os.pathsep+str(PACKAGE))
print('Paired release verified. Original source is read-only. New EXP015 outputs:',DEST)


In [ ]:
# 2. Install the pinned environment in an isolated venv; all processes use Agg.
subprocess.run([sys.executable,str(PACKAGE/'src/bootstrap_env.py'),'--package',str(PACKAGE),'--venv',str(VENV)],check=True,env=ENV)
PY=str(VENV/'bin/python')
subprocess.run([PY,'-c',"from integrity import configure; import json; print('PINNED RUNTIME',json.dumps(configure('cuda'),sort_keys=True))"],check=True,env=ENV)
print('Measured setup/install seconds:',round(time.monotonic()-SETUP_STARTED,2))


In [ ]:
# 3. Run numerical and recovery regressions. These are CPU/synthetic tests.
test_report=CACHE/'NOTEBOOK_CPU_REGRESSION.json';test_log=CACHE/'NOTEBOOK_CPU_REGRESSION.log'
result=subprocess.run([PY,'-u',str(PACKAGE/'tests/run_tests.py'),'--report',str(test_report),'--log',str(test_log)],env=ENV)
if result.returncode:
    print(test_log.read_text()[-18000:] if test_log.exists() else 'No test log was produced.')
    raise RuntimeError('CPU regression failed. Scientific NASA inference has not started.')
receipt=json.loads(test_report.read_text())
if receipt['status']!='PASS_CPU_REGRESSION':raise RuntimeError('Missing passing CPU receipt')
persist_verified(test_report,DEST/'engineering'/('CPU_REGRESSION_'+sha_file(test_report)[:16]+'.json'))
persist_verified(test_log,DEST/'engineering'/('CPU_REGRESSION_'+sha_file(test_log)[:16]+'.log'))
print('CPU regressions passed:',receipt['tests_run'],'tests in',round(receipt['seconds'],2),'seconds. Actual GPU gates follow.')


In [ ]:
# 4. Execute or hash-verify/resume the 18-fit, 729-stream inference-only supplement.
# This worker requires real full-shape GPU checks and bitwise original-output replay.
subprocess.run([PY,'-u',str(PACKAGE/'src/run.py'),'--source',str(SOURCE_PATH),'--work',str(WORK),'--backup',str(DEST),'--cache',str(CACHE)],check=True,env=ENV)


In [ ]:
# 5. Show the report and the durable immutable archive.
from IPython.display import display,Markdown,Image
latest=json.loads((DEST/'LATEST_EXPORT.json').read_text());final_archive=DEST/latest['archive']
if sha_file(final_archive)!=latest['sha256']:raise RuntimeError('Final Drive archive hash differs')
display(Markdown((WORK/'reports/REPORT_READY_SUMMARY.md').read_text()))
display(Image(filename=str(WORK/'reports/detector_comparison.png')))
display(Image(filename=str(WORK/'reports/pipeline_substitution.png')))
display(Image(filename=str(WORK/'reports/alarm_burden.png')))
print('VERIFIED FINAL ARCHIVE:',final_archive,'\nSHA-256:',latest['sha256'])
print('Send this archive back for independent analysis. No Overleaf or EXP012–EXP014 scientific output was edited.')


## Resume and interpretation

After a disconnect, use the same A100 capacity, run slot, paired release and original source, then **Run all**. Hash-verified Drive backups restore committed channels. Completed units, sealed scores and sealed reports are reused. The current unfinished channel and changes after the most recent durable backup may repeat. Keep one active runtime per slot. Code/input/environment mismatches stop recovery.

The reference is command-only; removal is entity-only. Changing the score representation and detector together does not isolate a threshold effect. Compare graph against both controls, strict point discrimination and alarm burden. Report equally improved controls and poor outcomes explicitly. Hundman's startup lookahead, unlabelled TEST adaptation and batch issuance are retained, including late notifications and misses. Historical TEST exposure and released TEST-extrema scaling remain limitations.

The final archive contains new reference/residual arrays, all score identities, replay and memory gates, evaluator outputs, per-seed and mission tables, timing/missed events, paired descriptive uncertainty, figures, frozen sources, scientific decision and final integrity audit. Read `START_HERE.md`, `docs/SCIENTIFIC_DECISION.md`, `docs/RECOVERY.md`, `docs/INPUTS.md` and `docs/TEST_REPORT.md` in the paired bundle.
